In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Fungsi Qiskit adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API IBM Quantum Platform). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.

## Gambaran keseluruhan
Dalam kimia kuantum, masalah struktur elektronik memfokuskan pada mencari penyelesaian kepada persamaan Schrödinger elektronik — fungsi gelombang kuantum yang menggambarkan tingkah laku elektron sistem. Fungsi gelombang ini adalah vektor amplitud kompleks, dengan setiap amplitud sepadan dengan sumbangan konfigurasi elektron yang mungkin.

Keadaan dasar adalah fungsi gelombang tenaga terendah sistem dan mempunyai kepentingan istimewa dalam kajian sistem molekul. Pendekatan paling tepat untuk mengira keadaan dasar mempertimbangkan semua konfigurasi elektron yang mungkin, tetapi ini menjadi tidak dapat diuruskan untuk sistem yang lebih besar kerana bilangan konfigurasi bertumbuh secara eksponen dengan saiz sistem.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) adalah kaedah hibrid kuantum-klasik yang inovatif untuk menganggar keadaan dasar sistem molekul dengan tepat. Ia mengintegrasikan perkakasan kuantum dengan pengkomputeran klasik, menggunakan pemproses kuantum untuk meneroka konfigurasi elektron calon dengan cekap dan mengira fungsi gelombang yang terhasil pada komputer klasik. Dengan menghasilkan fungsi gelombang yang padat namun tepat secara kimia, HI-VQE meningkatkan penyelidikan dan penemuan dalam kimia kuantum dan sains bahan.

![Image showing an overview of Qunova's HI-VQE algorithm](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE mengurangkan kerumitan pengkomputeran masalah struktur elektronik dengan menganggar keadaan dasar secara cekap dengan ketepatan tinggi. Ia memfokuskan pada subset konfigurasi elektron yang paling relevan yang dipilih dengan teliti, mengoptimumkan ketepatan dan kecekapan.

Menggabungkan kekuatan komputer klasik dan kuantum, HI-VQE secara berulang memperhalusi dan menambah baik anggaran fungsi gelombang semasa. Teknik pembinaan subspace yang unik membantu menjadikan pemilihan konfigurasi lebih cekap, supaya pengguna mempunyai kawalan pengkomputeran yang lebih besar dan ketepatan yang lebih baik dalam simulasi kimia kuantum.

Jika anda ingin mempelajari algoritma dengan lebih mendalam, anda boleh [membaca kertas penyelidikan berkaitan](https://arxiv.org/abs/2503.06292).
## Penerangan
Bilangan konfigurasi elektron untuk sistem molekul bertumbuh secara eksponen dengan saiz sistem. Walau bagaimanapun, untuk keadaan elektronik tertentu, seperti keadaan dasar, adalah lazim bahawa hanya sebahagian kecil konfigurasi menyumbang secara signifikan kepada tenaga keadaan. Kaedah Interaksi Konfigurasi Terpilih (SCI) memanfaatkan kekurpaan ini untuk mengurangkan kos pengkomputeran dengan mengenal pasti dan memfokuskan pada konfigurasi yang paling relevan. Subset konfigurasi ini dirujuk sebagai subspace.

HI-VQE memanfaatkan kecekapan intrinsik komputer kuantum untuk mewakili sistem molekul bagi membantu pencarian subspace. Ia mengintegrasikan subrutin klasik dan kuantum untuk menyelesaikan masalah struktur elektronik dengan ketepatan tinggi. Berbeza dengan kaedah SCI kuantum sedia ada, HI-VQE menggabungkan latihan variasi, pembinaan subspace berulang, dan penapisan konfigurasi pra-pengilangan untuk meningkatkan kecekapan dengan mengurangkan pengukuran kuantum, lelaran, dan kos pengilangan klasik. Oleh itu, HI-VQE boleh diaplikasikan kepada sistem molekul yang lebih besar yang memerlukan lebih banyak Qubit, dan mengurangkan kos untuk menyelesaikan masalah bersaiz tertentu kepada tahap ketepatan yang sama.

![Image showing a detailed description of each step in Qunova's HI-VQE algorithm.](../docs/images/guides/qunova-chemistry/description.avif)

Untuk mengira keadaan dasar sistem, HI-VQE mula-mula menggunakan pakej kimia klasik PySCF untuk menjana perwakilan molekul daripada input yang diberikan pengguna, seperti geometri molekul dan maklumat molekul lain. Ia kemudian memasuki gelung pengoptimuman hibrid kuantum-klasik, secara berulang memperhalusi subspace untuk mewakili keadaan dasar secara optimum sambil meminimumkan bilangan konfigurasi yang disertakan. Gelung berterusan sehingga kriteria penumpuan, seperti saiz subspace atau kestabilan tenaga, dipenuhi, selepas itu fungsi gelombang keadaan dasar dan tenaga yang dikira dikeluarkan. Keputusan ini boleh digunakan untuk membina permukaan tenaga potensial yang tepat dan melakukan analisis lanjutan sistem.

Gelung pengoptimuman memfokuskan pada melaraskan parameter Circuit kuantum untuk menghasilkan subspace berkualiti tinggi. HI-VQE menawarkan tiga pilihan Circuit kuantum: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), dan [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Pengoptimuman dimulakan dekat dengan keadaan rujukan Hartree-Fock kerana kesesuaiannya yang umum. Circuit kemudian dilaksanakan pada peranti kuantum dan konfigurasi disampling dari keadaan kuantum yang terhasil sebelum dikembalikan sebagai rentetan binari. Disebabkan hingar peranti kuantum, beberapa konfigurasi yang disampling mungkin tidak sah secara fizikal, gagal memulihara bilangan elektron atau spin. HI-VQE menangani ini menggunakan proses pemulihan konfigurasi dari pakej [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), supaya pengguna boleh sama ada membetulkan konfigurasi tidak sah atau membuangnya.

Konfigurasi yang sah kemudian melalui langkah penapisan pilihan untuk mengalih keluar yang dijangka menyumbang secara minimal. Ini mengurangkan dimensi subspace, seterusnya menurunkan kos langkah pengilangan. Jika penapisan diaktifkan, Hamiltonian subspace awal dibina dari konfigurasi yang sah dan pengilangan dilakukan dengan kriteria penamatan yang sangat longgar. Walaupun ketepatan amplitud yang terhasil untuk setiap konfigurasi adalah rendah, ia berkesan untuk meramalkan konfigurasi mana yang hendak ditinggalkan daripada subspace pada lelaran ini, dan ia cepat untuk dikira.

Konfigurasi yang dipilih ditambahkan ke subspace, dan Hamiltonian sistem diprojeksikan ke dalam subspace ini. Subspace dikemas kini secara berulang, mengekalkan konfigurasi yang paling relevan merentasi lelaran. Pendekatan ini berbeza dengan kaedah alternatif kerana Circuit kuantum tidak perlu menghampiri keadaan dasar penuh pada setiap langkah.

Seterusnya, Hamiltonian subspace digilang secara klasik untuk mendapatkan nilai eigen terendah dan eigenvector yang sepadan, mewakili penghampiran keadaan dasar dan tenaganya. Apabila kualiti subspace bertambah baik melalui lelaran, keadaan dasar yang dikira lebih menghampiri keadaan dasar sebenar. Langkah penapisan tambahan boleh dilakukan pada ketika ini untuk mengalih keluar konfigurasi daripada subspace yang tidak mempunyai sumbangan ketara kepada keadaan dasar yang dikira. Langkah ini memastikan subspace yang dibawa ke lelaran seterusnya adalah sepadat mungkin. Ini dinilai berdasarkan amplitud yang dikembalikan oleh pengilangan, kerana ini mewakili kepentingan sumbangan setiap konfigurasi kepada keadaan dasar yang dikira.

Semakan penumpuan kemudian menentukan sama ada latihan lanjut akan meningkatkan keputusan. Jika ya, langkah pengembangan klasik pilihan dilakukan, parameter Circuit kuantum dikemas kini untuk lebih meminimumkan tenaga yang dikira, dan proses berulang. Langkah pengembangan klasik menjana konfigurasi tambahan untuk subspace, melengkapi konfigurasi yang disampling dari peranti kuantum. Ia mula-mula mengenal pasti konfigurasi dengan amplitud terbesar dalam keputusan pengilangan, sebelum menjana konfigurasi baharu dengan eksitasi tunggal dan berganda dari konfigurasi yang dikenal pasti. Bilangan konfigurasi yang dikehendaki kemudian ditambahkan ke subspace.

Setelah ditentukan bahawa lelaran telah menumpu, HI-VQE mengembalikan keadaan dasar yang dikira (dalam bentuk keadaan dalam subspace dan amplitud mereka dalam fungsi gelombang keadaan dasar), tenaganya, dan ukuran varians tenaga yang memberi petunjuk sama ada keadaan yang dikira membentuk eigenstate Hamiltonian sistem.

Pengguna boleh menentukan Circuit kuantum yang digunakan dan bilangan shot yang diambil untuk setiap Circuit kuantum, serta mengawal saiz subspace atau membolehkan penjanaan klasik konfigurasi tambahan untuk membantu konfigurasi yang dijana kuantum. Dengan cara ini pengguna boleh menyesuaikan tingkah laku HI-VQE untuk memenuhi aplikasi yang dikehendaki.
## Mulakan
Pertama, [mohon akses ke fungsi](https://forms.office.com/r/zN3hvMTqJ1).
Kemudian, sahkan menggunakan [kunci API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) anda dan, dengan mengandaikan anda telah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan anda, pilih Fungsi Qiskit seperti berikut:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Input
Lihat jadual berikut untuk semua parameter input yang diterima fungsi. Bahagian [Pilihan molekul](#molecule-options) dan [Pilihan HI-VQE](#hi-vqe-options) yang berikutnya memberikan butiran lanjut tentang argumen tersebut.
| Nama                   | Jenis                                                           | Penerangan                                                                                                                                                                                                                                                                                                 | Diperlukan | Lalai                                                                  | Contoh                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Ini boleh berupa rentetan atau senarai berstruktur yang mengandungi pasangan atom dan koordinat. Jika diberikan sebagai rentetan, ia mestilah geometri molekul dalam Format Koordinat Kartesian. Jika diberikan sebagai senarai, ia hendaklah diberikan sebagai senarai senarai yang masing-masing mengandungi rentetan atom dan tuple koordinat. | Ya      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` or `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nama backend untuk membuat pertanyaan.                                                                                                                                                                                                                                                                      | Ya      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Dimensi subspace maksimum untuk pengilangan. Lebih sedikit keadaan akan digunakan jika nombor bukan kuasa dua sempurna.                                                                                                                                                                                                                                                    | Ya      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Bilangan maksimum keadaan CI yang dijana secara klasik untuk disertakan dalam setiap lelaran.                                                                                                                                                                                                                     | Ya      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Pilihan berkaitan molekul yang digunakan sebagai input kepada HI-VQE. Lihat bahagian [Pilihan molekul](#molecule-options) untuk butiran lanjut.                                                                                                                                                                                                                                                | Tidak       | Lihat bahagian [Pilihan molekul](#molecule-options) untuk butiran.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Pilihan yang mengawal tingkah laku algoritma HI-VQE. Lihat bahagian [Pilihan HI-VQE](#hi-vqe-options) untuk butiran lanjut.                                                                                                                                                                                                                                                | Tidak       | Lihat bahagian [Pilihan HI-VQE](#hi-vqe-options) untuk butiran.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### Pilihan molekul
Jadual berikut memperincikan semua kunci dan nilai yang boleh ditetapkan dalam kamus `molecule_options`, serta jenis data dan nilai lalainya. Semua kunci adalah pilihan.

| Kunci                               | Jenis nilai                          | Nilai lalai                    | Julat sah                                                                                                                                                    | Penjelasan                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Pelbagai                                                                                                                                                        | Integer yang menentukan jumlah cas bersih sistem molekul. Nilai lalai ialah 0; walau bagaimanapun, ia boleh menjadi sebarang integer.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Pelbagai                                                                                                                                                        | Rentetan yang menentukan jenis asas; ini dihantar ke pyscf. (contoh: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Setiap indeks orbital.             | Indeks orbital ruang yang sah untuk masalah tersebut.                                                                                                             | Senarai indeks orbital aktif dalam selang [0, n) di mana n adalah bilangan Qubit yang digunakan dalam masalah. Jika ini ditentukan, argumen frozen_orbitals juga mesti ditentukan.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | Tiada indeks.                      | Indeks orbital ruang yang sah untuk masalah tersebut, tidak termasuk orbital aktif.                                                                                  | Senarai indeks orbital beku dalam julat yang sama seperti active_orbitals. Jika ditentukan, active_orbitals juga mesti ditentukan. Perhatikan bahawa hanya orbital yang diduduki patut dibekukan kerana bilangan elektron aktif dikurangkan sebanyak 2 untuk setiap orbital yang diduduki yang dibekukan.                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbital molekul Hartree-Fock. | Pelbagai.                                                                                                                                                       | Pekali untuk orbital ruang yang digunakan dalam pengiraan integral tolakan elektronik sistem. Beberapa contoh yang sah ialah orbital molekul Hartree-Fock, orbital semula jadi, dan orbital AVAS.                                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` atau `False`                                                                                                                                              | Digunakan untuk mengaktifkan simetri kumpulan titik bagi pengiraan molekul awal untuk membina asas orbital yang disesuaikan simetri. Orbital yang disesuaikan simetri ini digunakan sebagai fungsi asas untuk pengiraan SCF berikutnya. Nilai lalai ialah False; jika ditetapkan kepada True, ia akan diaktifkan dan kumpulan titik sewenang-wenangnya akan dikesan dan digunakan secara automatik. Jika simetri tertentu ditetapkan, contohnya, symmetry = "Dooh", maka ralat akan dinaikkan jika geometri molekul tidak tertakluk kepada simetri yang diperlukan ini. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Boleh digunakan untuk menjana subkumpulan simetri yang dikesan. Ini tidak berkesan apabila simetri ditentukan menggunakan argumen kata kunci symmetry.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan unit pengukuran yang digunakan untuk koordinat dan jarak atom. Lalainya adalah menggunakan unit angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan model nuklear yang hendak digunakan. Secara lalai ia menggunakan model nuklear titik, dan nilai lain membolehkan model nuklear Gaussian. Jika fungsi diberikan, ia akan digunakan dengan model nuklear Gaussian untuk menjana nilai taburan cas nuklear 'zeta'.                                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan pseudopotential untuk atom dalam molekul. Ini adalah None secara lalai, menunjukkan bahawa tiada pseudopotential digunakan dan semua elektron disertakan secara eksplisit dalam pengiraan.                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan sama ada hendak menggunakan GTO kartesian sebagai fungsi asas momentum sudut dalam pengiraan. Nilai lalai False akan menggunakan GTO sfera.                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menetapkan momen magnetik spin kolinear setiap atom. Secara lalai, ini adalah None dan setiap atom dimulakan dengan spin sifar.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | cth. ["H 1s", "O 2p"] untuk H2O                                                                                                                                                             | Ini mentakrifkan Orbital Atom yang hendak disertakan dalam skim AVAS. Lihat [dokumentasi AVAS](https://arxiv.org/abs/1701.07862) .                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            |  Antara 0.0 dan 2.0                                                                                                                                                      |  Ini menentukan nilai potong yang digunakan untuk menentukan Orbital Atom (AO) mana yang dikekalkan dalam ruang aktif.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` atau `"ccsd"`                                                                                                                                            | Ini mentakrifkan pendekatan teori untuk menyediakan orbital semula jadi dan memilih orbital aktif berdasarkan skim Nombor Penghunian Orbital Semula Jadi (NOONs). Lihat [dokumentasi NOONs](https://doi.org/10.1063/5.0042147). Kedua-dua indeks orbital aktif dan beku mesti diberikan untuk mengawal bilangan orbital (dan bilangan Qubit).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### Pilihan HI-VQE
Jadual berikut memperincikan semua kunci dan nilai yang boleh ditetapkan dalam kamus `hivqe_options`, serta jenis data dan nilai lalainya. Semua kunci adalah pilihan.

| Kunci                               | Jenis nilai                          | Nilai lalai                    | Julat sah                                                                                                                                                    | Penjelasan                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Antara 1 dan 10 000.                                                                                                                                          | Bilangan shot yang digunakan pada peranti kuantum setiap lelaran.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Antara 1 dan 50.                                                                                                                                              | Bilangan maksimum lelaran yang dijalankan untuk mengoptimumkan ansatz. Algoritma mungkin menggunakan lebih sedikit lelaran jika penumpuan dicapai lebih awal.                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | Keadaan Hartree-Fock.          | Rentetan binari dengan bilangan bit yang sepadan dengan bilangan Qubit yang diperlukan untuk masalah tersebut.                                                                 | Boleh digunakan untuk memulakan semula algoritma dengan keadaan klasik dari keputusan sebelumnya.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, atau `"lucj"`                                                                                                                                  | Ini menentukan ansatz kuantum yang hendak dioptimumkan untuk menjana keadaan baharu. `"epa"` memilih ansatz pemeliharaan eksitasi. `"hea"` memilih ansatz cekap perkakasan. `"lucj"` memilih ansatz Jastrow kluster unitari setempat.                                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | Sekurang-kurangnya 2.                                                                                                                                                    | Bilangan lelaran tanpa perubahan signifikan tenaga yang dikira yang perlu berlalu sebelum algoritma dianggap telah menumpu.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Lebih daripada 0 dan paling banyak 1.                                                                                                                                     | Magnitud perubahan dalam tenaga yang dikira yang dianggap signifikan untuk tujuan semakan penumpuan.                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` atau `False`                                                                                                                                              | Jika `True`, lelaran `convergence_count` mesti berlaku tanpa perubahan signifikan yang mengganggu untuk layak sebagai menumpu. Jika `False`, algoritma akan berhenti selepas `convergence_count` jika perubahan tidak signifikan telah berlaku pada mana-mana lelaran semasa proses pengoptimuman.                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` atau `False`.                                                                                                                                             | Sama ada hendak menggunakan pemulihan konfigurasi dari pakej `qiskit-addon-sqd`. Jika True, keadaan tidak sah yang disampling dari peranti kuantum diperbetulkan secara klasik. Jika False, ia dibuang.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Mana-mana satu daripada `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, atau `"sca"`. Jika menggunakan ansatz `"lucj"`, `"lucj_default"` juga merupakan pilihan. | Ini menentukan skim belitan yang hendak digunakan dalam Circuit kuantum, mengikut konvensyen Qiskit dan [ffsim yang biasa untuk ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Lebih daripada 0.                                                                                                                                                | Bilangan pengulangan setiap lapisan dalam Circuit kuantum.                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Sekurang-kurangnya 0, dan kurang daripada 1.                                                                                                                                   | Toleran untuk menentukan keadaan mana yang patut ditapis daripada subspace selepas pengilangan. Ia menentukan ambang inklusi untuk keadaan subspace berdasarkan amplitud yang dikira.                                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Antara `1e-4` dan `1e-1`, inklusif.                                                                                                                          | Toleran untuk meramalkan keadaan mana yang patut ditapis daripada subspace sebelum pengilangan. Ia mengawal ketepatan amplitud yang diramalkan untuk setiap keadaan, dengan nilai yang lebih kecil menghasilkan ramalan yang lebih tepat.                                                                                                                                                                                                                                                                                            |
## Output
Fungsi mengembalikan kamus dengan empat kunci dan nilai. Kunci dan nilai didokumentasikan dalam jadual berikut:

| Kunci             | Jenis nilai    | Penjelasan                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Tenaga keadaan dasar anggaran molekul.                                                                      |
| `"states"`      | `List[str]`   | Determinan yang dipilih yang membentuk subspace yang digunakan untuk menyelesaikan tenaga. Ia dalam format alfa-beta berselang-seli. |
| `"eigenvector"` | `List[float]` | Eigenvector yang sepadan dengan keadaan dasar subspace yang terdiri daripada `"states"`.                                 |
| `"energy_variance"` | `float` | Varians tenaga keadaan dasar subspace yang terdiri daripada `"states"`, memberi petunjuk tentang kualiti penyelesaian. Nilai ini tidak negatif dan nilai yang lebih rendah bermakna keadaan dasar subspace lebih hampir menghampiri eigenstate Hamiltonian sistem. |
| `"energy_history"` | `List[float]` | Tenaga yang dikira setiap lelaran semasa proses pengoptimuman hibrid, dalam urutan yang sama ia dikira. Dua tenaga dikira setiap lelaran sebagai sebahagian daripada proses pengoptimuman SPSA. |
## Contoh
Contoh pertama menunjukkan cara mengira tenaga keadaan dasar untuk molekul NH3 menggunakan algoritma HI-VQE.
#### Takrifkan geometri molekul dan pilihan
Geometri molekul NH3 diberikan dengan koordinat Kartesian yang dipisahkan dengan ";" untuk setiap atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Pilihan tambahan boleh ditakrifkan dan diberikan untuk sistem molekul dalam format kamus berikut.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Laksanakan fungsi dengan input geometri dan pilihan.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Adalah baik untuk mencetak ID kerja Fungsi supaya ia boleh diberikan dalam permintaan sokongan jika ada yang tidak kena.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Contoh ini kemudian menggunakan 16 Qubit dengan 8 orbital asas sto3g untuk molekul NH3.
Semak [status](/guides/functions#check-job-status) atau kembalikan [keputusan](/guides/functions#retrieve-results) beban kerja Fungsi Qiskit anda seperti berikut:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Selepas kerja selesai, keputusan boleh diperoleh dengan instans `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Untuk mengakses tenaga keadaan dasar, gunakan kunci "energy". Kunci "eigenvector" menyediakan pekali CI dengan notasi bitstring konfigurasi elektron yang sepadan yang disimpan dengan "states" dalam keputusan.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Prestasi
Bahagian ini menunjukkan pengiraan penanda aras yang ditunjukkan HI-VQE dengan kes 24-Qubit untuk Li2S, kes 40-Qubit untuk molekul N2, dan kes 44-Qubit untuk sistem FeP-NO.
#### Lengkung permukaan tenaga potensial dissosiasi untuk molekul Li2S dengan 24 Qubit
Lengkung PES ditunjukkan dengan rujukan FCI dan teka-teki awal dari RHF, bersama-sama dengan ralat tenaga dari rujukan FCI.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Pengiraan telah dijalankan dengan geometri dan pilihan berikut.